In [1]:
# add_edge_features_4dags.py
# For each DAG (NOTEARS/PC/GES/GOLEM) in GEXF:
#   for each directed edge (u -> v) with weight w:
#       add feature: w * X[u] * X[v]
# Outputs (per DAG):
#   ./add_feature_dataset/<ALG>/training_data_normalized_with_edge_features_<ALG>.csv
#   ./add_feature_dataset/<ALG>/edge_feature_metadata_<ALG>.json
# Optional combined output (all DAGs merged):
#   ./add_feature_dataset/ALL/training_data_normalized_with_edge_features_ALL.csv
#   ./add_feature_dataset/ALL/edge_feature_metadata_ALL.json

import os
import re
import json
from typing import Dict, Any, List, Tuple

import pandas as pd
import networkx as nx


# =========================
# Config (edit if needed)
# =========================
DATA_PATH = "./training_data_normalized.csv"
OUT_BASE = "./add_feature_dataset"

# If you run locally, set these to your local paths:
GEXF_PATHS = {
    "NOTEARS": "./graph_NOTEARS.gexf",
    "PC": "./graph_PC.gexf",
    "GES": "./graph_GES.gexf",
    "GOLEM": "./graph_GOLEM.gexf",
}

# Make one merged dataset containing all DAG edge-features
MAKE_COMBINED_ALL = True

# Prefix for new columns
COL_PREFIX = "edge"

# If True, missing weight raises an error; if False, skip such edges.
STRICT_WEIGHT = True


# =========================
# Helpers
# =========================
def is_int_str(x: str) -> bool:
    return re.fullmatch(r"-?\d+", str(x)) is not None


def ensure_digraph(G) -> nx.DiGraph:
    if isinstance(G, nx.MultiDiGraph):
        # Collapse multiedges by keeping them separate via unique names later
        return G
    if isinstance(G, nx.DiGraph):
        return G
    # If loaded as undirected, convert to DiGraph preserving listed edges
    return nx.DiGraph(G)


def build_node_to_col_mapping(df: pd.DataFrame, G: nx.DiGraph) -> Dict[Any, str]:
    """
    Map GEXF node IDs to dataframe column names.

    Supported cases:
    1) Node IDs exactly match column names
    2) Node IDs are integer indices into df.columns
    3) Fallback: whitespace-stripped matching
    """
    cols = list(df.columns)
    nodes = list(G.nodes())

    # (1) exact match
    if all(str(n) in df.columns for n in nodes):
        return {n: str(n) for n in nodes}

    # (2) integer index mapping
    if all(is_int_str(n) for n in nodes):
        idxs = [int(str(n)) for n in nodes]
        if min(idxs) >= 0 and max(idxs) < len(cols):
            return {n: cols[int(str(n))] for n in nodes}

    # (3) whitespace-stripped fallback
    normalized_cols = {re.sub(r"\s+", "", c): c for c in cols}
    mapping: Dict[Any, str] = {}
    for n in nodes:
        key = re.sub(r"\s+", "", str(n))
        if key not in normalized_cols:
            mapping = {}
            break
        mapping[n] = normalized_cols[key]
    if mapping:
        return mapping

    missing = [str(n) for n in nodes if str(n) not in df.columns]
    raise ValueError(
        "Graph nodes do not match dataset columns, and auto-mapping failed.\n"
        f"- Example missing nodes: {missing[:10]}\n"
        f"- Dataset columns (first 20): {cols[:20]}\n"
        "Fix options:\n"
        "1) Rename GEXF nodes to match CSV column names, or\n"
        "2) Add a manual node->column mapping in code."
    )


def get_edge_weight(attrs: Dict[str, Any]) -> float:
    w = attrs.get("weight", None)
    if w is None:
        w = attrs.get("value", None)
    if w is None:
        raise KeyError("missing weight")
    return float(w)


def make_unique(name: str, existing: set) -> str:
    if name not in existing:
        return name
    i = 1
    while f"{name}__dup{i}" in existing:
        i += 1
    return f"{name}__dup{i}"


def iter_edges_with_attrs(G) -> List[Tuple[Any, Any, Dict[str, Any]]]:
    """
    Return list of (u, v, attrs) for DiGraph and MultiDiGraph.
    """
    if isinstance(G, nx.MultiDiGraph):
        out = []
        for u, v, k, attrs in G.edges(keys=True, data=True):
            # include key so we can disambiguate in naming if needed
            attrs2 = dict(attrs)
            attrs2["_multikey"] = str(k)
            out.append((u, v, attrs2))
        return out
    else:
        return list(G.edges(data=True))


def add_edge_features_for_alg(
    df_base: pd.DataFrame,
    alg: str,
    gexf_path: str,
    out_dir: str,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Return (df_with_features, metadata_dict)
    """
    df = df_base.copy()
    original_cols = list(df.columns)

    G = ensure_digraph(nx.read_gexf(gexf_path))
    node_to_col = build_node_to_col_mapping(df, G)

    existing_cols = set(df.columns)
    edges_meta: List[Dict[str, Any]] = []
    added_cols: List[str] = []

    for u, v, attrs in iter_edges_with_attrs(G):
        try:
            w = get_edge_weight(attrs)
        except Exception:
            if STRICT_WEIGHT:
                raise ValueError(f"[{alg}] Edge ({u}->{v}) has no numeric weight.")
            else:
                continue

        u_col = node_to_col[u]
        v_col = node_to_col[v]

        # base name includes alg to avoid collisions when merging
        base_name = f"{COL_PREFIX}_{alg}__{str(u)}__{str(v)}"
        if "_multikey" in attrs:
            base_name += f"__k{attrs['_multikey']}"

        new_col = make_unique(base_name, existing_cols)

        df[new_col] = w * df[u_col] * df[v_col]

        existing_cols.add(new_col)
        added_cols.append(new_col)
        edges_meta.append(
            {
                "alg": alg,
                "u": str(u),
                "v": str(v),
                "u_col": u_col,
                "v_col": v_col,
                "weight": w,
                "new_col": new_col,
            }
        )

    os.makedirs(out_dir, exist_ok=True)
    out_csv = os.path.join(out_dir, f"training_data_normalized_with_edge_features_{alg}.csv")
    df.to_csv(out_csv, index=False)

    meta = {
        "alg": alg,
        "data_path": DATA_PATH,
        "gexf_path": gexf_path,
        "output_csv": out_csv,
        "n_rows": int(df.shape[0]),
        "n_original_cols": int(len(original_cols)),
        "n_added_edge_features": int(len(added_cols)),
        "added_feature_cols": added_cols,
        "node_to_col_mapping_preview": {str(k): v for k, v in list(node_to_col.items())[:30]},
        "edges": edges_meta,
    }

    out_meta = os.path.join(out_dir, f"edge_feature_metadata_{alg}.json")
    with open(out_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"[DONE:{alg}] added={len(added_cols)} -> {out_csv}")
    return df, meta


# =========================
# Main
# =========================
def main():
    os.makedirs(OUT_BASE, exist_ok=True)

    df_base = pd.read_csv(DATA_PATH, low_memory=False)

    per_alg_results: Dict[str, Dict[str, Any]] = {}
    combined_df = df_base.copy()
    combined_existing = set(combined_df.columns)

    # Build per-ALG outputs
    for alg, gexf_path in GEXF_PATHS.items():
        out_dir = os.path.join(OUT_BASE, alg)
        df_alg, meta_alg = add_edge_features_for_alg(df_base, alg, gexf_path, out_dir)
        per_alg_results[alg] = meta_alg

        # Merge into combined (only new cols for this alg)
        if MAKE_COMBINED_ALL:
            new_cols = [c for c in df_alg.columns if c not in df_base.columns]
            # In case something collides (shouldn't, because alg is in name), keep unique
            for c in new_cols:
                if c in combined_existing:
                    # very unlikely, but safe
                    c2 = make_unique(c, combined_existing)
                    combined_df[c2] = df_alg[c].values
                    combined_existing.add(c2)
                else:
                    combined_df[c] = df_alg[c].values
                    combined_existing.add(c)

    # Save combined
    if MAKE_COMBINED_ALL:
        all_dir = os.path.join(OUT_BASE, "ALL")
        os.makedirs(all_dir, exist_ok=True)

        out_all_csv = os.path.join(all_dir, "training_data_normalized_with_edge_features_ALL.csv")
        combined_df.to_csv(out_all_csv, index=False)

        out_all_meta = os.path.join(all_dir, "edge_feature_metadata_ALL.json")
        with open(out_all_meta, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "data_path": DATA_PATH,
                    "output_csv": out_all_csv,
                    "n_rows": int(combined_df.shape[0]),
                    "n_original_cols": int(df_base.shape[1]),
                    "n_total_cols": int(combined_df.shape[1]),
                    "per_alg": per_alg_results,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(f"[DONE:ALL] total_cols={combined_df.shape[1]} -> {out_all_csv}")


if __name__ == "__main__":
    main()


[DONE:NOTEARS] added=9 -> ./add_feature_dataset\NOTEARS\training_data_normalized_with_edge_features_NOTEARS.csv
[DONE:PC] added=24 -> ./add_feature_dataset\PC\training_data_normalized_with_edge_features_PC.csv
[DONE:GES] added=55 -> ./add_feature_dataset\GES\training_data_normalized_with_edge_features_GES.csv
[DONE:GOLEM] added=11 -> ./add_feature_dataset\GOLEM\training_data_normalized_with_edge_features_GOLEM.csv


C:\Users\User\AppData\Local\Temp\ipykernel_18308\4166263379.py:249: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_df[c] = df_alg[c].values


[DONE:ALL] total_cols=114 -> ./add_feature_dataset\ALL\training_data_normalized_with_edge_features_ALL.csv
